In [1]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

True
Tesla T4


In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
!find "/content/drive/MyDrive" -type d -name "selected_transformer_candidate_a"

/content/drive/MyDrive/Accenture_October_Transformer/selected_transformer_candidate_a


In [4]:
from google.colab import files
uploaded = files.upload()

Saving validation_chunks_768_512.csv to validation_chunks_768_512.csv


In [5]:
import json
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from tqdm.auto import tqdm
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer
)

MODEL_DIR = Path(
    "/content/drive/MyDrive/Accenture_October_Transformer/"
    "selected_transformer_candidate_a"
)
with open(MODEL_DIR / "thresholds.json") as f:
    thresholds = json.load(f)

with open(MODEL_DIR / "category_order.json") as f:
    category_order = json.load(f)

with open(MODEL_DIR / "metadata.json") as f:
    metadata = json.load(f)

MAX_LENGTH = metadata["max_length"]

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = (
    AutoModelForSequenceClassification
    .from_pretrained(MODEL_DIR)
    .to(device)
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

model.eval()

print("Device:", device)
print("Categories:", len(category_order))
print("Max length:", MAX_LENGTH)

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

Device: cuda
Categories: 10
Max length: 4096


In [6]:
val_chunks = pd.read_csv(
    "/content/validation_chunks_768_512.csv"
)

print(val_chunks.shape)
val_chunks.head()

(2481, 15)


,chunk_id,title,config,config_version,chunk_size,overlap,step_size,chunk_index,token_start,token_end,token_count,char_start,char_end,text,text_hash
0,"WHITESMOKE,INC_11_08_2011-EX-10.26-PROMOTION A...","WHITESMOKE,INC_11_08_2011-EX-10.26-PROMOTION A...",768_512,gate3_chunking_v1,768,512,256,0,0,768,768,0,5105,Exhibit 10.26 CONFIDENTIAL TREATMENT HAS BE...,174931c8d44f6579f63d4a914e68075441dc4cca2be804...
1,"WHITESMOKE,INC_11_08_2011-EX-10.26-PROMOTION A...","WHITESMOKE,INC_11_08_2011-EX-10.26-PROMOTION A...",768_512,gate3_chunking_v1,768,512,256,1,256,1024,768,1714,6512,] during a [* ] (whether during the [* ] or a ...,a7b92600fc2aca41ee9080072bca96eee5b8f29d6d9312...
2,"WHITESMOKE,INC_11_08_2011-EX-10.26-PROMOTION A...","WHITESMOKE,INC_11_08_2011-EX-10.26-PROMOTION A...",768_512,gate3_chunking_v1,768,512,256,2,512,1280,768,3417,7826,negative response) given by the Criteria Check...,eee21d76ff31386b4abc5f51f416910b9171a8c7f87e27...
3,"WHITESMOKE,INC_11_08_2011-EX-10.26-PROMOTION A...","WHITESMOKE,INC_11_08_2011-EX-10.26-PROMOTION A...",768_512,gate3_chunking_v1,768,512,256,3,768,1536,768,5106,9543,Browser.\n\n (a) any parent company of that p...,e3522b959335f115c089b365adee17ad44be7ef0c7a58f...
4,"WHITESMOKE,INC_11_08_2011-EX-10.26-PROMOTION A...","WHITESMOKE,INC_11_08_2011-EX-10.26-PROMOTION A...",768_512,gate3_chunking_v1,768,512,256,4,1024,1792,768,6513,11193,the [ * ]. [ * ] only include those [ * ] whic...,3fe0794cfe6704ebbc3702c667c2d7333588854a3b1bb2...


In [7]:
BATCH_SIZE = 8

rows = []

model.eval()

for start in tqdm(
    range(0, len(val_chunks), BATCH_SIZE),
    desc="Candidate A validation inference"
):

    batch_df = val_chunks.iloc[
        start:start + BATCH_SIZE
    ]

    encoded = tokenizer(
        batch_df["text"].tolist(),
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

    encoded = {
        key: value.to(device)
        for key, value in encoded.items()
    }

    with torch.no_grad():

        outputs = model(**encoded)

        scores = torch.sigmoid(
            outputs.logits
        ).cpu().numpy()

    for j, (_, chunk) in enumerate(
        batch_df.iterrows()
    ):

        for i, category in enumerate(
            category_order
        ):

            score = float(scores[j, i])
            threshold = float(
                thresholds[category]
            )

            rows.append({
                "chunk_id": chunk["chunk_id"],
                "title": chunk["title"],
                "config": chunk["config"],
                "chunk_index": int(chunk["chunk_index"]),
                "char_start": int(chunk["char_start"]),
                "char_end": int(chunk["char_end"]),
                "text_hash": chunk["text_hash"],
                "category": category,
                "score": score,
                "threshold": threshold,
                "detection": int(
                    score >= threshold
                )
            })


candidate_a_validation_predictions = pd.DataFrame(
    rows
)

Candidate A validation inference:   0%|          | 0/311 [00:00<?, ?it/s]

In [8]:
print(
    "Rows:",
    len(candidate_a_validation_predictions)
)

print(
    "Unique chunks:",
    candidate_a_validation_predictions[
        "chunk_id"
    ].nunique()
)

print(
    "Outputs per chunk:",
    candidate_a_validation_predictions
    .groupby("chunk_id")
    .size()
    .unique()
)

print(
    "Finite scores:",
    np.isfinite(
        candidate_a_validation_predictions["score"]
    ).all()
)

print(
    "Detection check:",
    (
        candidate_a_validation_predictions["detection"]
        ==
        (
            candidate_a_validation_predictions["score"]
            >=
            candidate_a_validation_predictions["threshold"]
        ).astype(int)
    ).all()
)

Rows: 24810
Unique chunks: 2481
Outputs per chunk: [10]
Finite scores: True
Detection check: True


In [9]:
ARTIFACT_DIR = Path(
    "/content/drive/MyDrive/"
    "Accenture_October_Transformer/"
    "validation_artifacts"
)

ARTIFACT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

prediction_path = (
    ARTIFACT_DIR
    / "candidate_a_validation_predictions.csv"
)

candidate_a_validation_predictions.to_csv(
    prediction_path,
    index=False
)

print("Saved to:")
print(prediction_path)

Saved to:
/content/drive/MyDrive/Accenture_October_Transformer/validation_artifacts/candidate_a_validation_predictions.csv


In [10]:
import hashlib

sha = hashlib.sha256()

with open(prediction_path, "rb") as f:
    for block in iter(
        lambda: f.read(1024 * 1024),
        b""
    ):
        sha.update(block)

prediction_hash = sha.hexdigest()

print("SHA-256:")
print(prediction_hash)

SHA-256:
c28e3dd829aac4b4a1bdbcb05d23bae2a56de8afb508cd130623703a52f8182f
